In [6]:
pip install tensorflow-hub

  Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl.metadata (1.3 kB)
  Using cached tf_keras-2.20.1-py3-none-any.whl.metadata (1.8 kB)
Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl (30 kB)
Using cached tf_keras-2.20.1-py3-none-any.whl (1.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tensorflow-hub]
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# FedPIGD-B v2 (FULL WORKING VERSION)
# Backbone: DenseNet121 (native tf.keras, stable)
# Your novelty preserved: GA pipelines + signatures + anchoring + FL
# ============================================================

import os
import time
import copy
import random
import gc
import numpy as np
import tensorflow as tf

from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize

# ============================================================
# 0) CONFIG
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRAIN_DIR = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/train_data"
TEST_DIR  = "/Users/tanvir/Downloads/Research/Fish Recognition/Photos/Fish/test_data"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
LR = 1e-4
DROPOUT = 0.3

NUM_CLIENTS = 4
FL_ROUNDS = 10

EV_POP = 10
EV_GENS = 6
EV_SUBSET = 64
MAX_PIPE_LEN = 5

ANCHOR_LAMBDA = 1e-3

# ============================================================
# LOGGING
# ============================================================
def ts(): return time.strftime("%H:%M:%S")
def log(msg): print(f"[{ts()}] [INFO] {msg}")
def phase(msg):
    print("\n" + "=" * 80)
    print(msg)
    print("=" * 80)

def cleanup():
    gc.collect()

# ============================================================
# 1) DATA LOADING
# ============================================================
def list_classes(train_dir):
    classes = sorted(d for d in os.listdir(train_dir)
                     if os.path.isdir(os.path.join(train_dir, d)))
    return classes, {c: i for i, c in enumerate(classes)}

def load_paths_labels(data_dir, class_to_idx):
    paths, labels = [], []
    for c, i in class_to_idx.items():
        cls_dir = os.path.join(data_dir, c)
        for f in os.listdir(cls_dir):
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                paths.append(os.path.join(cls_dir, f))
                labels.append(i)
    return np.array(paths), np.array(labels)

def split_clients(paths, labels, n):
    idx = np.random.permutation(len(paths))
    paths, labels = paths[idx], labels[idx]
    proportions = np.random.dirichlet([1.0] * n)
    sizes = (proportions * len(paths)).astype(int)
    sizes[-1] = len(paths) - sum(sizes[:-1])

    out, s = [], 0
    for i, sz in enumerate(sizes):
        out.append((paths[s:s+sz], labels[s:s+sz]))
        log(f"Client C{i+1}: {sz} samples")
        s += sz
    return out

phase("Loading dataset")
class_names, class_to_idx = list_classes(TRAIN_DIR)
K = len(class_names)

train_paths, train_labels = load_paths_labels(TRAIN_DIR, class_to_idx)
test_paths,  test_labels  = load_paths_labels(TEST_DIR,  class_to_idx)

clients = split_clients(train_paths, train_labels, NUM_CLIENTS)

# ============================================================
# 2) PIL PIPELINE OPS (non-differentiable)
# ============================================================
OP_SPACE = [
    "IDENTITY", "GRAYSCALE", "AUTO_CONTRAST", "EQUALIZE",
    "SHARPEN", "BRIGHTNESS", "CONTRAST", "BLUR"
]

def sample_op():
    op = random.choice(OP_SPACE)
    p = {}
    if op in ["SHARPEN", "BRIGHTNESS", "CONTRAST"]:
        p["factor"] = float(np.random.uniform(0.8, 1.5))
    if op == "BLUR":
        p["radius"] = float(np.random.uniform(0.2, 1.5))
    return (op, p)

def apply_op(img, op):
    name, p = op
    if name == "IDENTITY":
        return img
    if name == "GRAYSCALE":
        return ImageOps.grayscale(img).convert("RGB")
    if name == "AUTO_CONTRAST":
        return ImageOps.autocontrast(img)
    if name == "EQUALIZE":
        return ImageOps.equalize(img)
    if name == "SHARPEN":
        return ImageEnhance.Sharpness(img).enhance(p["factor"])
    if name == "BRIGHTNESS":
        return ImageEnhance.Brightness(img).enhance(p["factor"])
    if name == "CONTRAST":
        return ImageEnhance.Contrast(img).enhance(p["factor"])
    if name == "BLUR":
        return img.filter(ImageFilter.GaussianBlur(p["radius"]))
    return img

def apply_pipeline_np(x, pipe):
    # x is float32 [0,1]
    img = Image.fromarray((np.clip(x, 0.0, 1.0) * 255).astype(np.uint8))
    for op in pipe:
        img = apply_op(img, op)
    return np.array(img).astype(np.float32) / 255.0

def tf_apply_pipeline(x, pipe):
    out = tf.numpy_function(lambda z: apply_pipeline_np(z, pipe), [x], tf.float32)
    out.set_shape([IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    return out

# ============================================================
# 3) SAFE DECODE
# ============================================================
def decode_and_resize(p):
    img = tf.image.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE)
    return tf.cast(img, tf.float32) / 255.0

# ============================================================
# 4) DENSENET121 MODEL (frozen) + adapter + head
# ============================================================
def build_model(num_classes):
    base = tf.keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=(224, 224, 3)
    )
    base.trainable = False

    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dense(128, activation="relu", name="adapter")(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)
    out = tf.keras.layers.Dense(num_classes, activation="softmax", name="head")(x)

    model = tf.keras.Model(inputs=base.input, outputs=out)
    return model, base

def build_feat_model(backbone):
    return tf.keras.Model(
        inputs=backbone.input,
        outputs=tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
    )

# ✅ instantiate global model + feature extractor (missing in your paste)
global_model, backbone = build_model(K)
feat_model = build_feat_model(backbone)

# ============================================================
# 5) EVOLUTION (SEPARABILITY PROXY)
# ============================================================
def separability_proxy(f, l):
    sb, sw = 0.0, 0.0
    mu = f.mean(axis=0, keepdims=True)
    for c in np.unique(l):
        Xc = f[l == c]
        if len(Xc) < 2:
            continue
        mc = Xc.mean(axis=0, keepdims=True)
        sb += len(Xc) * np.sum((mc - mu) ** 2)
        sw += np.sum((Xc - mc) ** 2)
    return sb / (sw + 1e-8)

def evolve_pipeline(paths, labels):
    pop = [[sample_op() for _ in range(random.randint(1, MAX_PIPE_LEN))]
           for _ in range(EV_POP)]

    best_pipe, best_score = None, -1e9

    for g in range(EV_GENS):
        for pipe in pop:
            idx = np.random.choice(len(paths), min(EV_SUBSET, len(paths)), replace=False)
            xs = []
            for p in paths[idx]:
                x = decode_and_resize(tf.constant(p))
                x = tf_apply_pipeline(x, pipe)
                xs.append(x.numpy())
            xs = np.array(xs, dtype=np.float32)

            feats = feat_model.predict(xs, verbose=0)
            score = separability_proxy(feats, labels[idx]) - 0.02 * len(pipe)

            if score > best_score:
                best_score = score
                best_pipe = pipe

            del xs, feats
            cleanup()

        log(f"GA Gen {g+1}/{EV_GENS} | BestScore={best_score:.4f}")
    return best_pipe

phase("Evolving pipelines")
client_pipelines = [evolve_pipeline(cp, cl) for cp, cl in clients]

# ============================================================
# 6) PIPELINE SIGNATURES
# ============================================================
def pipeline_signature(pipe):
    hist = np.zeros(len(OP_SPACE), dtype=np.float32)
    factors, radii = [], []
    for (op, p) in pipe:
        hist[OP_SPACE.index(op)] += 1.0
        if "factor" in p: factors.append(float(p["factor"]))
        if "radius" in p: radii.append(float(p["radius"]))
    f_mean = np.mean(factors) if factors else 0.0
    f_std  = np.std(factors) if factors else 0.0
    r_mean = np.mean(radii) if radii else 0.0
    r_std  = np.std(radii) if radii else 0.0
    length = float(len(pipe))
    runtime_proxy = length
    return np.concatenate([hist, np.array([f_mean, f_std, r_mean, r_std, length, runtime_proxy], np.float32)])

client_sigs = np.stack([pipeline_signature(p) for p in client_pipelines], axis=0)
log(f"Signature dim = {client_sigs.shape[1]}")
log(f"Signatures shape = {client_sigs.shape}")

# ============================================================
# 7) FEDERATED TRAINING + SIGNATURE-CONDITIONED ANCHORING
# ============================================================
def get_adapter_vector(model):
    w = model.get_layer("adapter").get_weights()
    return np.concatenate([w[0].ravel(), w[1].ravel()], axis=0).astype(np.float32)

def get_trainable_tensors(model):
    return model.get_layer("adapter").weights + model.get_layer("head").weights

def extract_trainable_numpy(model):
    return [w.numpy() for w in get_trainable_tensors(model)]

def assign_trainable_numpy(model, new_vals):
    for var, val in zip(get_trainable_tensors(model), new_vals):
        var.assign(val)

def fedavg(trainable_sets, sizes):
    total = float(sum(sizes))
    avg = []
    for wi in range(len(trainable_sets[0])):
        acc = 0.0
        for ci in range(len(trainable_sets)):
            acc += (sizes[ci] / total) * trainable_sets[ci][wi]
        avg.append(acc)
    return avg

class ConditionerRidge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.models = None
    def fit(self, S, A):
        self.models = [Ridge(alpha=self.alpha).fit(S, A[:, j]) for j in range(A.shape[1])]
    def predict(self, s):
        s = s.reshape(1, -1)
        return np.array([m.predict(s)[0] for m in self.models], dtype=np.float32)

conditioner = ConditionerRidge(alpha=1.0)
anchors = [get_adapter_vector(global_model).copy() for _ in range(NUM_CLIENTS)]
log("Anchors initialized from global adapter vector.")

def make_client_dataset(paths, labels, pipeline, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(paths)), seed=SEED, reshuffle_each_iteration=True)

    def map_fn(x, y, pipe=pipeline):
        img = decode_and_resize(x)
        img = tf_apply_pipeline(img, pipe)
        return img, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def train_one_epoch_with_anchor(model, dataset, optimizer, anchor_vec, lam, client_id, round_id):
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
    adapter_layer = model.get_layer("adapter")
    kernel_var, bias_var = adapter_layer.weights

    anchor_tf = tf.convert_to_tensor(anchor_vec, dtype=tf.float32)

    total_loss = 0.0
    steps = 0

    for xb, yb in dataset:
        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            ce = loss_fn(yb, probs)

            k_flat = tf.reshape(kernel_var, [-1])
            b_flat = tf.reshape(bias_var, [-1])
            cur_vec = tf.concat([k_flat, b_flat], axis=0)

            anchor_loss = tf.reduce_sum(tf.square(cur_vec - anchor_tf))
            loss = ce + lam * anchor_loss

        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))

        total_loss += float(loss.numpy())
        steps += 1

        if steps % 20 == 0:
            log(f"[Round {round_id}] C{client_id} batch={steps} | loss={total_loss/steps:.4f}")

    return total_loss / max(steps, 1)

phase("Federated training + anchoring")

for r in range(1, FL_ROUNDS + 1):
    phase(f"FL ROUND {r}/{FL_ROUNDS}")

    global_weights = global_model.get_weights()
    local_trainables = []
    sizes = []
    local_adapter_centers = []

    for ci, (cp, cl) in enumerate(clients, start=1):
        log(f"Client C{ci}: build local model + sync global weights")
        local_model, _ = build_model(K)
        local_model.set_weights(global_weights)

        ds = make_client_dataset(cp, cl, client_pipelines[ci-1], shuffle=True)

        opt = tf.keras.optimizers.Adam(LR)
        avg_loss = train_one_epoch_with_anchor(
            local_model, ds, opt,
            anchor_vec=anchors[ci-1],
            lam=ANCHOR_LAMBDA,
            client_id=ci,
            round_id=r
        )
        log(f"Client C{ci}: epoch done | avg loss={avg_loss:.4f}")

        local_trainables.append(extract_trainable_numpy(local_model))
        sizes.append(len(cp))
        local_adapter_centers.append(get_adapter_vector(local_model))

        del ds, opt
        cleanup()

    log("Server: FedAvg aggregate adapter+head updates")
    avg_trainables = fedavg(local_trainables, sizes)
    assign_trainable_numpy(global_model, avg_trainables)
    log("Server: global model updated (adapter+head).")

    log("Server: update conditioner (signature -> adapter anchor)")
    A = np.stack(local_adapter_centers, axis=0)
    conditioner.fit(client_sigs, A)
    anchors = [conditioner.predict(client_sigs[i]) for i in range(NUM_CLIENTS)]
    log("Server: anchors refreshed for next round.")

    del local_trainables, local_adapter_centers, avg_trainables, A
    cleanup()

# ============================================================
# FINAL EVALUATION (ALL METRICS + FULL LOGS)
# ============================================================
phase("FINAL EVALUATION: Test set (no client preprocessing)")

def make_test_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def map_fn(x, y):
        img = decode_and_resize(x)
        return img, y

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds = make_test_dataset(test_paths, test_labels)

pred_probs = global_model.predict(test_ds, verbose=1)
preds = np.argmax(pred_probs, axis=1)

acc  = accuracy_score(test_labels, preds)
prec = precision_score(test_labels, preds, average="weighted", zero_division=0)
rec  = recall_score(test_labels, preds, average="weighted", zero_division=0)
f1   = f1_score(test_labels, preds, average="weighted", zero_division=0)
ll   = log_loss(test_labels, pred_probs)

y_true_bin = label_binarize(test_labels, classes=list(range(K)))

roc_auc = roc_auc_score(
    y_true_bin,
    pred_probs,
    average="macro",
    multi_class="ovr"
)

pr_auc = average_precision_score(
    y_true_bin,
    pred_probs,
    average="macro"
)

mcc = matthews_corrcoef(test_labels, preds)

log(f"Accuracy     = {acc:.4f}")
log(f"Precision    = {prec:.4f} (weighted)")
log(f"Recall       = {rec:.4f} (weighted)")
log(f"F1-score     = {f1:.4f} (weighted)")
log(f"ROC-AUC      = {roc_auc:.4f} (macro, OvR)")
log(f"PR-AUC       = {pr_auc:.4f} (macro)")
log(f"Log-Loss     = {ll:.4f}")
log(f"MCC Score    = {mcc:.4f}")

print("\nClassification Report:")
print(classification_report(
    test_labels,
    preds,
    target_names=class_names,
    digits=4,
    zero_division=0
))

cleanup()



Loading dataset
[11:34:05] [INFO] Client C1: 2648 samples
[11:34:05] [INFO] Client C2: 158 samples
[11:34:05] [INFO] Client C3: 1364 samples
[11:34:05] [INFO] Client C4: 765 samples
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 16s 1us/step

Evolving pipelines
[11:34:49] [INFO] GA Gen 1/6 | BestScore=1.1363
[11:35:18] [INFO] GA Gen 2/6 | BestScore=1.1363
[11:35:46] [INFO] GA Gen 3/6 | BestScore=1.3541
[11:36:11] [INFO] GA Gen 4/6 | BestScore=1.3541
[11:36:38] [INFO] GA Gen 5/6 | BestScore=1.3541
[11:37:03] [INFO] GA Gen 6/6 | BestScore=1.3551
[11:37:32] [INFO] GA Gen 1/6 | BestScore=0.9005
[11:38:00] [INFO] GA Gen 2/6 | BestScore=1.0330
[11:38:27] [INFO] GA Gen 3/6 | BestScore=1.0330
[11:38:54] [INFO] GA Gen 4/6 | BestScore=1.0330
[11:39:21] [INFO] GA Gen 5/6 | BestScore=1.0330
[11:39:47] [INFO] GA Gen 6/6 | BestScore=1.0778
[11:40:13] [INFO] GA Gen 1/6 | BestScore=1.1762
[11:40:38] [INFO] GA Gen 2/6 | BestScore=1.1762
[11:41:02] [INFO] GA Gen 3/6 | BestScore=1.1762
[11:41:27] [INFO] GA Gen 

2026-01-04 11:52:16.339914: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[11:52:16] [INFO] Client C1: epoch done | avg loss=2.7346
[11:52:16] [INFO] Client C2: build local model + sync global weights


2026-01-04 11:52:42.713839: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[11:52:42] [INFO] Client C2: epoch done | avg loss=3.6238
[11:52:42] [INFO] Client C3: build local model + sync global weights
[11:53:29] [INFO] [Round 1] C3 batch=20 | loss=3.3960
[11:54:14] [INFO] [Round 1] C3 batch=40 | loss=3.2954
[11:55:00] [INFO] [Round 1] C3 batch=60 | loss=3.1828
[11:55:46] [INFO] [Round 1] C3 batch=80 | loss=3.0895
[11:55:59] [INFO] Client C3: epoch done | avg loss=3.0685
[11:55:59] [INFO] Client C4: build local model + sync global weights
[11:56:47] [INFO] [Round 1] C4 batch=20 | loss=3.5156
[11:57:33] [INFO] [Round 1] C4 batch=40 | loss=3.2853


...

[12:10:21] [INFO] Client C4: epoch done | avg loss=2.5132
[12:10:21] [INFO] Server: FedAvg aggregate adapter+head updates
[12:10:21] [INFO] Server: global model updated (adapter+head).
[12:10:21] [INFO] Server: update conditioner (signature -> adapter anchor)
[12:10:55] [INFO] Server: anchors refreshed for next round.

FL ROUND 3/10
[12:10:55] [INFO] Client C1: build local model + sync global weights
[12:11:44] [INFO] [Round 3] C1 batch=20 | loss=2.0704
[12:12:31] [INFO] [Round 3] C1 batch=40 | loss=2.0145
[12:13:17] [INFO] [Round 3] C1 batch=60 | loss=1.9453
[12:14:09] [INFO] [Round 3] C1 batch=80 | loss=1.9298
[12:14:53] [INFO] [Round 3] C1 batch=100 | loss=1.9078
[12:15:37] [INFO] [Round 3] C1 batch=120 | loss=1.8718
[12:16:21] [INFO] [Round 3] C1 batch=140 | loss=1.8421
[12:17:04] [INFO] [Round 3] C1 batch=160 | loss=1.8265
[12:17:16] [INFO] Client C1: epoch done | avg loss=1.8146
[12:17:17] [INFO] Client C2: build local model + sync global weights
[12:17:39] [INFO] Client C2: epoc

2026-01-04 12:34:36.698231: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[12:34:36] [INFO] Client C4: epoch done | avg loss=1.8213
[12:34:36] [INFO] Server: FedAvg aggregate adapter+head updates
[12:34:36] [INFO] Server: global model updated (adapter+head).
[12:34:36] [INFO] Server: update conditioner (signature -> adapter anchor)
[12:35:10] [INFO] Server: anchors refreshed for next round.

FL ROUND 5/10
[12:35:10] [INFO] Client C1: build local model + sync global weights
[12:35:56] [INFO] [Round 5] C1 batch=20 | loss=1.5037
[12:36:40] [INFO] [Round 5] C1 batch=40 | loss=1.4659
[12:37:24] [INFO] [Round 5] C1 batch=60 | loss=1.4147
[12:38:09] [INFO] [Round 5] C1 batch=80 | loss=1.4109
[12:38:53] [INFO] [Round 5] C1 batch=100 | loss=1.4033
[12:39:38] [INFO] [Round 5] C1 batch=120 | loss=1.3699
[12:40:23] [INFO] [Round 5] C1 batch=140 | loss=1.3544
[12:41:08] [INFO] [Round 5] C1 batch=160 | loss=1.3508
[12:41:20] [INFO] Client C1: epoch done | avg loss=1.3378
[12:41:20] [INFO] Client C2: build local model + sync global weights
[12:41:43] [INFO] Client C2: epoc

2026-01-04 17:13:20.450042: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[17:13:20] [INFO] Client C4: epoch done | avg loss=1.1047
[17:13:20] [INFO] Server: FedAvg aggregate adapter+head updates
[17:13:20] [INFO] Server: global model updated (adapter+head).
[17:13:20] [INFO] Server: update conditioner (signature -> adapter anchor)
[17:13:54] [INFO] Server: anchors refreshed for next round.

FL ROUND 9/10
[17:13:54] [INFO] Client C1: build local model + sync global weights
[17:14:40] [INFO] [Round 9] C1 batch=20 | loss=0.8685
[17:15:24] [INFO] [Round 9] C1 batch=40 | loss=0.8416
[17:16:08] [INFO] [Round 9] C1 batch=60 | loss=0.8248
[17:16:54] [INFO] [Round 9] C1 batch=80 | loss=0.8281
[17:17:41] [INFO] [Round 9] C1 batch=100 | loss=0.8511
[17:18:29] [INFO] [Round 9] C1 batch=120 | loss=0.8381
[17:19:13] [INFO] [Round 9] C1 batch=140 | loss=0.8242
[17:19:57] [INFO] [Round 9] C1 batch=160 | loss=0.8280
[17:20:09] [INFO] Client C1: epoch done | avg loss=0.8216
[17:20:10] [INFO] Client C2: build local model + sync global weights
[17:20:32] [INFO] Client C2: epoc